# TD2: Tokenization et embedding avec Hugging Face

**Objectifs**:
1. Charger et tester différents tokenizer de HuggingFace
2. Utiliser différents modèlse d'embeddings neuronaux pour projeter des textes
3. Analyser les résultats des modèles

#### Pré-requis

Lire le tutoriel ici: https://huggingface.co/learn/llm-course/en/chapter2/4

In [3]:
def clean_up_text(texte_brut):
    marque_debut = "*** START OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
    marque_fin = "*** END OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"

    texte = texte_brut.split(marque_debut, 1)[1]
    texte = texte.split(marque_fin, 1)[0].strip()

    return texte

## Téléchargements des livres

from pathlib import Path
from urllib.request import Request, urlopen

URLS = [
    "https://www.gutenberg.org/cache/epub/14155/pg14155.txt",
    "https://www.gutenberg.org/cache/epub/48881/pg48881.txt",
    "https://www.gutenberg.org/cache/epub/56528/pg56528.txt",
    "https://www.gutenberg.org/cache/epub/8600/pg8600.txt"
]

textes = []

for url in URLS:
    fichier = Path(f"corpus/{url.split("/")[-1]}")

    if not fichier.exists():
        fichier.parent.mkdir(parents=True, exist_ok=True)
        requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(requete) as reponse:
            contenu = reponse.read().decode("utf-8-sig")
        fichier.write_text(clean_up_text(contenu), encoding="utf-8")

    texte_brut = fichier.read_text(encoding="utf-8")
    print(f"{len(texte_brut):,} caractères chargés")

HTTPError: HTTP Error 503: Service Unavailable

#### Exercice 1: Utiliser les tokenizers de HuggingFace


In [4]:
# Exemple de chargement et d'utilisation d'un tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Questions**
1. Tokenisez les textes fournis en entrée à partir de la méthode `tokenize`.
2. Réalisez l'encodage des textes en utilisant le tokenizer `bert-base-cased` et la méthode `convert_tokens_to_ids`.
3. Décodez le texte à partir de l'encodage et comparez le avec le texte d'origine `decode`.
4. Comparez le nombre total de tokens dans le texte par rapport au nombre de mots.
5. Réalisez les mêmes opérations avec le tokenizer `camembert-base` et comparez les sorties.

#### Exercice 2: Utiliser des modèles d'embeddings neuronaux



In [9]:
## Exemple d'embedding (au niveau du token)
from transformers import pipeline, AutoTokenizer

extracteur = pipeline("feature-extraction", model="camembert-base")
tokenizer = AutoTokenizer.from_pretrained("camembert-base")

Device set to use mps:0


In [18]:
phrase = "C'est facile de réaliser un embedding!"
tokens = tokenizer.tokenize(phrase)
emb = extracteur("C'est facile de réaliser un embedding!")[0]

for tok, tok_vector in zip(tokens, emb):
    print(f"Représentation du token {tok} = {tok_vector}")
    print("--")


Représentation du token ▁C = [-0.007913392037153244, -0.020152896642684937, 0.07343368232250214, -0.11447037756443024, -0.04228593781590462, -0.012857258319854736, 0.012017173692584038, 0.22938881814479828, 0.00208471342921257, 0.09868142008781433, 0.029739398509263992, 0.0761914998292923, -0.04492921009659767, 0.12053517252206802, 0.24882113933563232, -0.0039819227531552315, 0.04384646564722061, -0.07904346287250519, 0.06440870463848114, -0.17919263243675232, 0.05884205177426338, -0.03674786537885666, -0.012001343071460724, -0.3280918002128601, 0.17983950674533844, -0.2740662097930908, -0.019579291343688965, -0.08329232782125473, -0.01634105294942856, 0.00620654970407486, 0.02532072737812996, -0.22775205969810486, 0.14966881275177002, 0.1296570599079132, 0.15633651614189148, -0.059449534863233566, -0.07845209538936615, 0.18008454144001007, -0.06865207850933075, 0.054768577218055725, -0.21668416261672974, 0.11015299707651138, 0.1474921554327011, -0.09109918773174286, 0.1552836745977401

/Users/sophrobhayek/Documents/dev/Sorbonne_Universite_courses/venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `CamembertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


1. Écrivez la fonction `word_representation` qui réalise la moyenne des embeddings du token pour obtenir une représentation par mot (définit comme étant l'ensemble des tokens obtenus lorsqu'on découpe sur l'espace).
2. Donner une représentation vectorielle pour chacun des mots du corpus.
3. Changez de granularité et donnez une représentation vectorielle de chaque chunk de 200 tokens.
4. Réalisez la même opération cette fois avec la bibliothèque `sentence-transformers`: https://sbert.net

#### Exercice 3: Analyse des résultats

1. Est-il possible de distinguer pour un mot donné quel auteur/roman l'emploie? N'hésitez pas à mobiliser les concepts que vous avez vu dans d'autres cours.
2. Répondez à la même question mais au niveau de la phrase.
3. Réalisez une représentation en 2D ou en 3D des embeddings (vous pouvez par exemple utiliser des méthodes issues de `https://scikit-learn.org/stable/modules/unsupervised_reduction.html`).